In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import fit_lib as fit
import BM29_lib as BM29
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams["text.usetex"] = False
from scipy import optimize
from scipy.optimize import curve_fit

# successful fits: 
# LV4 + aSyn WT                 
# LV4 + aSyn E61V
# LV4 + aSyn A124T
# LV4 + aSyn E46K
# POPG + aSyn WT
# POPG + 25uM aSyn variants
# DOPC/GM1 + aSyn WT
# DOPC/GM1 + 25uM aSyn variants
# DOPC/DOPS + aSyn WT
# DOPC/DOPS + 25uM aSyn variants

# Observables:
# 1. Median radius
# 2. Fraction of P(r) shifted to higher values (integral)

# presentation: 


In [ ]:
fit.lognormal_polydisp()

In [ ]:
import numpy as np
from scipy.stats import lognorm


def compare_lognormals(
    sigma_small, scale_small,
    sigma_large, scale_large,
    amplitude_small=1.0,
    amplitude_large=1.0
):
    """
    Find the intersections of two log-normal distributions and calculate
    the area between them for radii larger than the largest intersection.

    Parameters
    ----------
    sigma_small, sigma_large : float
        Shape parameters (sigma) of the log-normal distributions.
    scale_small, scale_large : float
        Scale parameters of scipy.stats.lognorm.
        For loc=0, these correspond to the medians of the distributions.
    amplitude_small, amplitude_large : float
        Optional multiplicative factors for the distributions.

    Returns
    -------
    dict
        intersection_radii:
            All positive intersection radii.
        tail_start:
            Largest intersection radius.
        signed_integral:
            Integral of (large distribution - small distribution)
            from tail_start to infinity.
        area_between_curves:
            Absolute area between the curves over the same interval.
    """

    ss, sl = sigma_small, sigma_large
    ms, ml = np.log(scale_small), np.log(scale_large)

    if ss <= 0 or sl <= 0 or scale_small <= 0 or scale_large <= 0:
        raise ValueError("Sigma and scale parameters must be positive.")

    # Coefficients of the quadratic equation in y = ln(r),
    # obtained by setting the two log-normal PDFs equal.
    a = 1 / (2 * ss**2) - 1 / (2 * sl**2)
    b = -ms / ss**2 + ml / sl**2
    c = (
        ms**2 / (2 * ss**2)
        - ml**2 / (2 * sl**2)
        + np.log(amplitude_large / amplitude_small)
        + np.log(ss / sl)
    )

    # Solve the quadratic equation.
    tol = 1e-12 * max(abs(a), abs(b), abs(c), 1.0)

    if abs(a) < tol:
        if abs(b) < tol:
            raise ValueError("The distributions are identical or have no unique intersection.")
        roots = [-c / b]
    else:
        discriminant = b**2 - 4 * a * c
        disc_tol = 1e-12 * max(b**2, abs(4 * a * c), 1.0)

        if discriminant < -disc_tol:
            raise ValueError("The distributions do not intersect.")

        discriminant = max(discriminant, 0.0)
        sqrt_disc = np.sqrt(discriminant)

        q = -0.5 * (b + np.copysign(sqrt_disc, b))

        if q == 0:
            roots = [-b / (2 * a)]
        else:
            roots = [q / a, c / q]

    # Convert from ln(r) to radii and remove duplicate intersections.
    intersection_radii = np.sort(np.exp(roots))
    intersection_radii = np.unique(np.round(intersection_radii, 12))

    if len(intersection_radii) == 0 or not np.all(np.isfinite(intersection_radii)):
        raise ValueError("No finite positive intersection found.")

    # Use the largest intersection: beyond this point, the curves
    # do not intersect again.
    r_cross = intersection_radii[-1]

    # Calculate the integrated difference using survival functions.
    sf_large = amplitude_large * lognorm.sf(
        r_cross, s=sl, scale=scale_large
    )
    sf_small = amplitude_small * lognorm.sf(
        r_cross, s=ss, scale=scale_small
    )

    signed_integral = sf_large - sf_small

    return {
        "intersection_radii": intersection_radii,
        "tail_start": r_cross,
        "signed_integral": signed_integral,
        "area_between_curves": abs(signed_integral)
    }

In [ ]:
result = compare_lognormals(
    sigma_small=0.35,
    scale_small=100,
    sigma_large=0.35,
    scale_large=130
)
print("Intersection:", result["tail_start"])
print("Signed integral:", result["signed_integral"])
print("Area between curves:", result["area_between_curves"])